# H SAF on CF0360: super-obs vs thinning in the cycling runs

Examples from the two H SAF-only cycling runs (2019-05-01 to 05-11; SMOS, SMAP, MODIS and CYGNSS monitored only),
with the open loop that monitors H SAF in raw mode as the reference:

| label | experiment | H SAF setting |
|---|---|---|
| **raw (OL)** | `M21C_test_CF0360_OL_monitor_HSAF` | raw mode, monitored only (no DA) |
| **super-obs** | `M21C_test_CF0360_sob025fp_HSAF_xc03125` | `superob_grid_deg = 0.25`, Gaussian footprint C = 12.5²·I + S (`9624356`); obs scaling derived for the **old** super-obs |
| **super-obs, OL sc.** | `M21C_test_CF0360_sob025fp_OL1sc_HSAF_xc03125` | the same super-obs, with the raw-obs scaling of the OL and the thinning run (rerun after section 1.1) |
| **thin 25 km** | `M21C_test_CF0360_thin25km_HSAF_xc03125` | `thin_dist_km = 25` (`5e220ff`: great-circle distance; `1f0e9ef` later switched to degrees) |

Two days: **2019-05-01** (first day; all runs start from the same restart) and **2019-05-10** (after nine days
of cycling). For each: what obs each run assimilated, their footprints, the weight each footprint puts on the CF0360
tiles, and the analysis increments that result. Every obs, position and predicted forecast is taken from, or checked
against, the runs' own ObsFcstAna (section 1).

**Data** (fetched once from Discover with `projects/M21C_land/scripts/fetch_cycling_superob_vs_thin.py` into
`~/Desktop/M21C_land_local/cf0360_cycling_sob_vs_thin/`): per run and window, ObsFcstAna and the ensemble-mean
`SFMC_FCST` / `SFMC_ANA`; raw H121 for 5/10 (5/1 is in `cf0360_obs_fcst_20190501/`). The OL-scaling super-ob rerun was
fetched the same way into `~/Desktop/M21C_land_local/cf0360_cycling_sob025fp_OL1sc/`. Runs locally (`regrid` env)
in about 3.5 minutes.

## 0. Settings and setup

In [ ]:
import datetime as dt
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import netCDF4 as nc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from scipy import sparse
from scipy.spatial import cKDTree

%matplotlib inline
T_START = time.time()

LOCAL = Path('/Users/amfox/Desktop/M21C_land_local')
NEW = LOCAL / 'cf0360_cycling_sob_vs_thin'
H121_DIRS = [LOCAL / 'cf0360_obs_fcst_20190501' / 'cf0360_obs_fcst_20190501' / 'H121', NEW / 'H121']
TILE_RASTER = LOCAL / 'superob_fov' / 'tile_raster_kansas.npz'      # 35.9-39.1N, 101.1-97.4W
NEW_OL1 = LOCAL / 'cf0360_cycling_sob025fp_OL1sc'
RUNS = {'raw (OL)': 'OL_monitor', 'super-obs': 'sob025fp', 'super-obs, OL sc.': 'sob025fp_OL1sc', 'thin 25 km': 'thin25km'}
RUN_DIR = {'OL_monitor': NEW / 'OL_monitor', 'sob025fp': NEW / 'sob025fp', 'thin25km': NEW / 'thin25km',
           'sob025fp_OL1sc': NEW_OL1 / 'sob025fp_OL1sc'}
RCOL = {'raw (OL)': '0.35', 'super-obs': 'tab:green', 'super-obs, OL sc.': 'tab:orange', 'thin 25 km': 'tab:purple'}
is_so = lambda lab: lab.startswith('super-obs')
WINDOWS = ([dt.datetime(2019, 5, 1, h) for h in range(3, 24, 3)]
           + [dt.datetime(2019, 5, 10, h) for h in range(0, 24, 3)])
DAYS = [dt.date(2019, 5, 1), dt.date(2019, 5, 10)]
SUPEROB_GRID = 0.25
THIN_KM = 25.0
REGION = (36.5, 38.5, -100.5, -98.0)       # SW Kansas, for maps (inside the tile raster)

def find_repo_root(start: Path) -> Path:
    for p in [start] + list(start.parents):
        if (p / '.git').exists():
            return p
    return start

REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / 'common' / 'python' / 'io'))
from read_GEOSldas import read_tilecoord
TILECOORD = Path('/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/land_sweeper/LS_OLv8_M36/output/'
                 'SMAP_EASEv2_M36_GLOBAL/rc_out/M21C_test_CF0360.ldas_tilecoord.bin')
FIG_DIR = REPO_ROOT / 'projects' / 'M21C_land' / 'figures' / 'cycling_superob_vs_thinning'
FIG_DIR.mkdir(parents=True, exist_ok=True)

SATS = ['metop_a', 'metop_b', 'metop_c']
SAT_LABEL = {'metop_a': 'Metop-A', 'metop_b': 'Metop-B', 'metop_c': 'Metop-C'}
DT_ASSIM = dt.timedelta(hours=3)
FOV_RAW_KM = 12.5
MAPL_RADIUS_KM = 6371.0
KM_PER_DEG = np.pi / 180.0 * MAPL_RADIUS_KM
KM2DEG = 1.0 / KM_PER_DEG
FAC_SEARCH = 2.0
N_LAND = 1188020

### Tiles, reader, tile assignment, footprints, tile shapes

Copied from `cf0360_superob_vs_thinning_obsspace.ipynb` and `cf0360_hsaf_superob_fov_v2.ipynb` (single-precision
details kept): reader QC and window, obs-to-tile assignment, the raw-obs footprint R (12.5 km Gaussian, 25 km
cut-off, used for raw mode and thinning) and the super-ob footprint N (Gaussian with C = 12.5²·I + S, q ≤ 4), and the
tile-raster drawing.

In [ ]:
tc = read_tilecoord(str(TILECOORD))
COM_LON = tc['com_lon'].astype(np.float32)
COM_LAT = tc['com_lat'].astype(np.float32)
BBOX = {k: tc[k].astype(np.float32) for k in ['min_lon', 'max_lon', 'min_lat', 'max_lat']}
AREA = tc['area'].astype(np.float64)
TREE = cKDTree(np.c_[COM_LON, COM_LAT].astype(np.float64))
assert np.abs(np.load(NEW / 'OL_monitor' / 'tile_lat.npy') - tc['com_lat']).max() < 1e-4, 'tile table differs from the runs'
CELL = (tc['j_indg'].astype(np.int64) - 1) * 360 + tc['i_indg'].astype(np.int64)      # cube cell of each land tile

TT_MINUS_UTC = 64.184
UNIX0 = dt.datetime(1970, 1, 1)

def read_h121(f, sat, w_low, w_up):
    with nc.Dataset(f) as d:
        d.set_auto_maskandscale(False)
        v = {k: d[k][:] for k in ['latitude', 'longitude', 'time', 'surface_soil_moisture', 'surface_flag',
                                  'processing_flag', 'backscatter40_flag', 'wetland_fraction',
                                  'topographic_complexity', 'subsurface_scattering_probability',
                                  'surface_soil_moisture_sensitivity']}
    s8 = lambda a: a.astype(np.int8).astype(np.int32)
    ssm = v['surface_soil_moisture'].astype(np.int32)
    t = v['time'] * 86400.0
    ok = ((ssm >= 0) & (ssm <= 10000) & (t > w_low) & (t <= w_up)
          & ((v['surface_flag'] & 1) == 0) & ((v['processing_flag'] & 3) == 0)
          & ((v['backscatter40_flag'] & 4) == 0)
          & (s8(v['wetland_fraction']) >= 0) & (s8(v['wetland_fraction']) < 10)
          & (s8(v['topographic_complexity']) >= 0) & (s8(v['topographic_complexity']) < 10)
          & (s8(v['subsurface_scattering_probability']) < 5)
          & (v['surface_soil_moisture_sensitivity'] * 1e-7 > 1.0))
    return pd.DataFrame({'sat': sat, 'lat': v['latitude'][ok].astype(np.float32) * np.float32(1e-6),
                         'lon': v['longitude'][ok].astype(np.float32) * np.float32(1e-6),
                         'sm': (ssm[ok] * 0.01 / 100).astype(np.float32)})

def read_window(t):
    lo = (t - DT_ASSIM / 2 - UNIX0).total_seconds() + TT_MINUS_UTC
    return pd.concat([read_h121(f, s, lo, lo + DT_ASSIM.total_seconds()) for s in SATS for f in window_files(s, t)],
                     ignore_index=True)                      # file order within each satellite

NLON_P, NLAT_P = 1440, 1080
DLON_P, DLAT_P = np.float32(360 / NLON_P), np.float32(180 / NLAT_P)

def pert_cell(lat, lon):
    i = np.ceil((np.asarray(lon, np.float32) - np.float32(-180)) / DLON_P).astype(np.int64)
    j = np.ceil((np.asarray(lat, np.float32) - np.float32(-90)) / DLAT_P).astype(np.int64)
    return np.clip(j, 1, NLAT_P) * NLON_P + np.clip(i, 1, NLON_P)

_key = pert_cell(COM_LAT, COM_LON)
_order = np.argsort(_key, kind='stable')
_sorted_key = _key[_order]

def assign_tiles(lat, lon, fov_km=FOV_RAW_KM):
    """1-based land tile per obs (get_tile_num_for_obs); -1 no tile in pert cell, -2 dropped"""
    lat, lon = np.asarray(lat, np.float32), np.asarray(lon, np.float32)
    k = pert_cell(lat, lon)
    a, b = np.searchsorted(_sorted_key, k), np.searchsorted(_sorted_key, k, side='right')
    t = np.full(len(lat), -1); dx = np.full(len(lat), np.inf, np.float32); dy = dx.copy()
    for n in np.flatnonzero(b > a):
        cand = _order[a[n]:b[n]]
        ddx, ddy = np.abs(lon[n] - COM_LON[cand]), np.abs(lat[n] - COM_LAT[cand])
        best = np.argmin(ddx + ddy)
        t[n], dx[n], dy[n] = cand[best], ddx[best], ddy[best]
    tt = np.clip(t, 0, None)
    outside = ((lon < BBOX['min_lon'][tt]) | (lon > BBOX['max_lon'][tt]) |
               (lat < BBOX['min_lat'][tt]) | (lat > BBOX['max_lat'][tt]))
    mdy = np.float32(fov_km * KM2DEG)
    mdx = (mdy / np.cos(np.radians(lat))).astype(np.float32)
    return np.where(t < 0, -1, np.where(outside & ((dx > mdx) | (dy > mdy)), -2, t + 1))

def _candidates(lat, lon, rx, ry):
    cand = TREE.query_ball_point(np.c_[lon, lat].astype(np.float64), r=np.maximum(rx, ry))
    obs = np.repeat(np.arange(len(lat)), [len(c) for c in cand])
    tile = np.concatenate(cand).astype(np.int64) if len(obs) else np.zeros(0, np.int64)
    dlon = COM_LON[tile].astype(np.float64) - lon[obs]
    dlat = COM_LAT[tile].astype(np.float64) - lat[obs]
    return obs, tile, dlon, dlat, (dlon / rx[obs]) ** 2 + (dlat / ry[obs]) ** 2

def _to_matrix(obs, tile, w, n):
    W = sparse.csr_matrix((w, (obs, tile)), shape=(n, N_LAND))
    s = np.asarray(W.sum(1)).ravel()
    return sparse.diags(np.where(s > 0, 1 / np.where(s > 0, s, 1), 0)) @ W

def fov_R(lat, lon, fov_km=FOV_RAW_KM):
    lat, lon = np.asarray(lat, np.float64), np.asarray(lon, np.float64)
    ry = np.full(len(lat), FAC_SEARCH * fov_km * KM2DEG); rx = ry / np.cos(np.radians(lat))
    obs, tile, _, _, nd2 = _candidates(lat, lon, rx, ry)
    k = nd2 <= 1.0
    return _to_matrix(obs[k], tile[k], np.exp(-0.5 * FAC_SEARCH ** 2 * nd2[k]) * AREA[tile[k]], len(lat))

def fov_N(lat, lon, S, fov_km=FOV_RAW_KM):
    lat, lon, S = np.asarray(lat, np.float64), np.asarray(lon, np.float64), np.asarray(S, np.float64)
    cxx, cyy, cxy = fov_km ** 2 + S[:, 0], fov_km ** 2 + S[:, 1], S[:, 2]
    det = cxx * cyy - cxy ** 2
    lmax = 0.5 * (cxx + cyy) + np.sqrt(0.25 * (cxx - cyy) ** 2 + cxy ** 2)
    ry = FAC_SEARCH * np.sqrt(lmax) * KM2DEG; rx = ry / np.cos(np.radians(lat))
    obs, tile, dlon, dlat, nd2 = _candidates(lat, lon, rx, ry)
    k = nd2 <= 1.0
    obs, tile, dlon, dlat = obs[k], tile[k], dlon[k], dlat[k]
    dx, dy = dlon * KM_PER_DEG * np.cos(np.radians(lat[obs])), dlat * KM_PER_DEG
    q = (cyy[obs] * dx ** 2 - 2 * cxy[obs] * dx * dy + cxx[obs] * dy ** 2) / det[obs]
    k = q <= FAC_SEARCH ** 2
    return _to_matrix(obs[k], tile[k], np.exp(-0.5 * q[k]) * AREA[tile[k]], len(lat))

def tile_average(df):
    """reader accumulation: one obs per (sat, tile): mean sm, lat, lon of the obs assigned to that tile"""
    k = df[df.tile > 0]
    key = k.sat.map({s: i for i, s in enumerate(SATS)}).to_numpy().astype(np.int64) * (N_LAND + 1) + k.tile.to_numpy()
    uk, inv = np.unique(key, return_inverse=True)
    n = np.bincount(inv)
    m = lambda x: np.bincount(inv, np.asarray(x, np.float64)) / n
    return pd.DataFrame({'sat': np.array(SATS)[uk // (N_LAND + 1)], 'tile': uk % (N_LAND + 1),
                         'lat': m(k.lat), 'lon': m(k.lon), 'sm': m(k.sm), 'n_raw': n})

def superobs(raw, grid=SUPEROB_GRID):
    """super-obs per (sat, cell): mean sm and position, spread S [km^2], tile (12.5 km), reader's collision rule"""
    n_lon, n_lat = int(round(360 / grid)), int(round(180 / grid))
    lon = np.mod(raw.lon.to_numpy(np.float64) + 180.0, 360.0) - 180.0
    lat = raw.lat.to_numpy(np.float64)
    i = np.minimum(n_lon, ((lon + 180.0) / grid).astype(int) + 1)
    j = np.minimum(n_lat, ((lat + 90.0) / grid).astype(int) + 1)
    sat_id = raw.sat.map({s: k for k, s in enumerate(SATS)}).to_numpy()
    uk, inv = np.unique((sat_id.astype(np.int64) * (n_lat + 1) + j) * (n_lon + 1) + i, return_inverse=True)
    N = np.bincount(inv)
    dlon, dlat = lon - (-180.0 + (i - 0.5) * grid), lat - (-90.0 + (j - 0.5) * grid)
    s = lambda x: np.bincount(inv, x) / N
    so = pd.DataFrame({'sat': np.array(SATS)[uk // ((n_lat + 1) * (n_lon + 1))],
                       'lat': s(lat).astype(np.float32), 'lon': s(lon).astype(np.float32),
                       'sm': s(raw.sm.to_numpy(np.float64)), 'n_raw': N})
    mdlon, mdlat = s(dlon), s(dlat)
    coslat = np.cos(np.pi / 180.0 * so.lat.to_numpy(np.float64))
    so['Sxx'] = np.maximum(0.0, s(dlon * dlon) - mdlon ** 2) * (KM_PER_DEG * coslat) ** 2
    so['Syy'] = np.maximum(0.0, s(dlat * dlat) - mdlat ** 2) * KM_PER_DEG ** 2
    so['Sxy'] = (s(dlon * dlat) - mdlon * mdlat) * KM_PER_DEG ** 2 * coslat
    so['tile'] = assign_tiles(so.lat, so.lon)
    t0 = np.clip(so.tile.to_numpy() - 1, 0, None)
    dl = np.abs(so.lon.to_numpy(np.float64) - COM_LON[t0].astype(np.float64)); dl = np.minimum(dl, 360 - dl)
    d2 = dl ** 2 + (so.lat.to_numpy(np.float64) - COM_LAT[t0].astype(np.float64)) ** 2
    holder = {}
    for n, (sat, t, dd) in enumerate(zip(so.sat.to_numpy(), so.tile.to_numpy(), d2)):
        if t > 0:
            h = holder.get((sat, t))
            if h is None or dd < np.float64(np.float32(h[1])):
                holder[sat, t] = (n, dd)
    keep = np.zeros(len(so), bool); keep[[n for n, _ in holder.values()]] = True
    return so[keep].reset_index(drop=True)

def to_xyz(lat, lon):
    la, lo = np.radians(np.asarray(lat, float)), np.radians(np.asarray(lon, float))
    return np.column_stack((np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)))

def thin_min_dist_5e220ff(lat, lon, r_km):
    """greedy: visit obs in order, keep unless closer than r_km (great-circle) to an obs kept earlier, as GEOSldas
    thin_min_dist of 5e220ff, which the thin25km run used. 1f0e9ef measures the distance in degrees instead; if the run
    is redone with 1f0e9ef, use thin_min_dist from hsaf_obs_thinning_explained.ipynb."""
    n = len(lat); keep = np.zeros(n, bool)
    if n == 0:
        return keep
    p = to_xyz(lat, lon); chord = 2 * np.sin(r_km / MAPL_RADIUS_KM / 2)
    pr = cKDTree(p).query_pairs(chord, output_type='ndarray')
    if len(pr):
        pr = pr[((p[pr[:, 0]] - p[pr[:, 1]]) ** 2).sum(1) < chord ** 2]
    E = sparse.csr_matrix((np.ones(len(pr), bool), (pr.max(1), pr.min(1))), shape=(n, n))
    for k in range(n):
        keep[k] = not keep[E.indices[E.indptr[k]:E.indptr[k + 1]]].any()
    return keep

def nn_km(lat, lon):
    if len(lat) < 2:
        return np.full(len(lat), np.nan)
    p = to_xyz(lat, lon); d, _ = cKDTree(p).query(p, 2)
    return 2 * np.arcsin(d[:, 1] / 2) * MAPL_RADIUS_KM

def window_files(sat, t):
    out = []
    for d in H121_DIRS:
        for f in (d / sat).glob('*.nc'):
            start = dt.datetime.strptime(f.name.split('_')[-6], '%Y%m%d%H%M%S')
            if t - DT_ASSIM / 2 - dt.timedelta(hours=1) < start <= t + DT_ASSIM / 2:
                out.append(f)
    return sorted(out, key=lambda f: f.name.split('_')[-6])      # sensing-start order, as the reader

def read_ofa(run, t):
    f = sorted(RUN_DIR[run].glob(f'*ObsFcstAna.{t:%Y%m%d_%H}00z.nc4'))[0]
    with nc.Dataset(f) as d:
        df = pd.DataFrame({v: np.asarray(d[v][:]) for v in ['species', 'tilenum', 'lat', 'lon', 'obs', 'obsvar',
                                                             'fcst', 'fcstvar', 'ana', 'assim_flag']})
        v = d['obsparam_descr']
        descr = [str(x).strip() for x in (nc.chartostring(v[:]) if v.ndim > 1 else v[:])]
        errstd = np.asarray(d['obsparam_errstd'][:])
    sat_of = {k + 1: f'metop_{s[-4].lower()}' for k, s in enumerate(descr) if s.startswith('ASCAT_HSAF_MET')}
    err = {sat_of[k]: errstd[k - 1] / 100 for k in sat_of}       # [% saturation] -> fraction
    df = df[df.species.isin(sat_of)].copy()
    df['sat'] = df.species.map(sat_of)
    df['OmF'] = df.obs - df.fcst
    return df.reset_index(drop=True), err

def load_field(run, t, v='fcst'):
    return np.load(RUN_DIR[run] / f'sfmc_{v}_{t:%Y%m%d_%H}00z.npy').astype(np.float64)

# ---- tile shapes from the raster window (pixel value = 1-based tile number; land tiles come first)
_r = np.load(TILE_RASTER)
RAS_TILE, RAS_LAT_E, RAS_LON_E = _r['tile'], _r['lat_e'], _r['lon_e']
CELL_OF_TILE = np.r_[-1, (tc['j_indg'].astype(np.int64) - 1) * 360 + tc['i_indg'].astype(np.int64)]

def raster_window(box):
    a, b, c, d = box
    r0, r1 = np.searchsorted(RAS_LAT_E, a, 'right') - 1, np.searchsorted(RAS_LAT_E, b, 'left')
    c0, c1 = np.searchsorted(RAS_LON_E, c, 'right') - 1, np.searchsorted(RAS_LON_E, d, 'left')
    return RAS_TILE[r0:r1, c0:c1], RAS_LAT_E[r0:r1 + 1], RAS_LON_E[c0:c1 + 1]

def edge_segments(a, lat_e, lon_e):
    r, c = np.nonzero(a[:, 1:] != a[:, :-1])
    v = np.stack([np.c_[lon_e[c + 1], lat_e[r]], np.c_[lon_e[c + 1], lat_e[r + 1]]], axis=1)
    r, c = np.nonzero(a[1:, :] != a[:-1, :])
    h = np.stack([np.c_[lon_e[c], lat_e[r + 1]], np.c_[lon_e[c + 1], lat_e[r + 1]]], axis=1)
    return np.concatenate([v, h])

def tile_background(ax, win, edge_lw=0.5):
    # land (light grey) / water (light blue), tile edges (light grey) and CF0360 cube-cell edges (dark grey)
    tile, lat_e, lon_e = win
    land = tile <= N_LAND
    ax.pcolormesh(lon_e, lat_e, np.where(land, 0, 1), cmap=mcolors.ListedColormap(['0.93', '#cfe3f3']),
                  vmin=-0.5, vmax=1.5, rasterized=True, zorder=0.5)
    cell = np.where(land, CELL_OF_TILE[np.where(land, tile, 0)], -1)
    for ids, color, lw in [(np.where(land, tile, -1), '0.72', edge_lw), (cell, '0.25', 1.4 * edge_lw)]:
        ax.add_collection(LineCollection(edge_segments(ids, lat_e, lon_e), colors=color, lw=lw,
                                         capstyle='projecting', zorder=1.5))

def paint_tiles(ax, win, tiles, values, **kw):
    # colour land tiles (0-based tile index) by value on the raster window
    tile, lat_e, lon_e = win
    lut = np.full(N_LAND + 1, np.nan)
    lut[np.asarray(tiles) + 1] = values
    land = tile <= N_LAND
    vmap = np.where(land, lut[np.where(land, tile, 0)], np.nan)
    return ax.pcolormesh(lon_e, lat_e, np.ma.masked_invalid(vmap), rasterized=True, zorder=1.0, **kw)

def finish(ax, box):
    a, b, c, d = box
    ax.set_xlim(c, d); ax.set_ylim(a, b)
    ax.set_aspect(1 / np.cos(np.radians(0.5 * (a + b))))
    ax.tick_params(labelsize=8)

def inbox(df, box):
    a, b, c, d = box
    return df[df.lat.between(a, b) & df.lon.between(c, d)]

def ellipse_xy(C2, nsig, n=120):
    ev, evec = np.linalg.eigh(C2)
    t = np.linspace(0, 2 * np.pi, n)
    return (evec @ (nsig * np.sqrt(ev)[:, None] * np.vstack([np.cos(t), np.sin(t)]))).T

def footprint_outline(r, kind):
    """outline of the footprint's support (lon, lat): raw/thin = 25 km cut-off circle, super-ob = q = 4 ellipse"""
    if kind == 'N':
        C2 = FOV_RAW_KM ** 2 * np.eye(2) + np.array([[r.Sxx, r.Sxy], [r.Sxy, r.Syy]])
        e = ellipse_xy(C2, FAC_SEARCH)
    else:
        t = np.linspace(0, 2 * np.pi, 120)
        e = FAC_SEARCH * FOV_RAW_KM * np.c_[np.cos(t), np.sin(t)]
    return r.lon + e[:, 0] * KM2DEG / np.cos(np.radians(r.lat)), r.lat + e[:, 1] * KM2DEG

## 1. Rebuild each run's obs and check them against its ObsFcstAna

For every window, from the raw H121 obs (reader QC):

- **raw:** average per (satellite, tile); match to the OL's ObsFcstAna; predict with R at the ObsFcstAna position;
- **thinning:** obs with a tile, greedy 25 km thinning in file order per satellite, then average per tile; predict with R;
- **super-obs:** average per 0.25° cell at the mean position, spread S, tile at the mean position, collision rule;
  predict with N.

Each run's ObsFcstAna holds only the obs that passed its model-side screening, so the check is that **every**
ObsFcstAna obs is found offline at the same position, and that the offline footprint applied to that run's own
`SFMC_FCST` reproduces its `fcst`.

In [ ]:
RAW, SOF, OBS, CHK, FCST, ANA, ERR = {}, {}, {r: [] for r in RUNS}, [], {}, {}, {}
for t in WINDOWS:
    raw = read_window(t)
    raw['tile'] = assign_tiles(raw.lat, raw.lon)
    RAW[t] = raw
    off = {'raw (OL)': tile_average(raw)}
    raw['kept'] = False
    for sat in SATS:
        k = raw[(raw.sat == sat) & (raw.tile > 0)]                     # file order, obs with a tile only
        keep = thin_min_dist_5e220ff(k.lat.to_numpy(np.float64), k.lon.to_numpy(np.float64), THIN_KM)
        raw.loc[k.index[keep], 'kept'] = True
    off['thin 25 km'] = tile_average(raw[raw.kept])
    off['super-obs'] = SOF[t] = superobs(raw)
    for lab, run in RUNS.items():
        FCST[lab, t] = load_field(run, t, 'fcst')
        ANA[lab, t] = load_field(run, t, 'ana')
        ofa, ERR[lab] = read_ofa(run, t)
        cfg = 'super-obs' if is_so(lab) else lab
        o = off[cfg][off[cfg].tile > 0]
        m = ofa.merge(o, how='left', left_on=['sat', 'tilenum'], right_on=['sat', 'tile'], suffixes=('', '_off'))
        if is_so(lab):
            W = fov_N(m.lat, m.lon, m[['Sxx', 'Syy', 'Sxy']].fillna(0).to_numpy())
        else:
            W = fov_R(m.lat, m.lon)
        m['F_off'] = np.where(m.n_raw.notna(), W @ FCST[lab, t], np.nan)      # only where found offline
        m['window'] = t
        dpos = np.hypot(m.lat - m.lat_off, m.lon - m.lon_off)
        dF = np.abs(m.F_off - m.fcst)
        CHK.append({'window': f'{t:%m-%d %H}z', 'run': lab, 'obs in ObsFcstAna': len(m),
                    'found offline': int(m.n_raw.notna().sum()), 'built offline': len(o),
                    'position diff > 1e-4°': int((dpos > 1e-4).sum()),
                    'max |F - fcst|': dF.max(), 'median |F - fcst|': dF.median()})
        OBS[lab].append(m)
OBS = {k: pd.concat(v, ignore_index=True) for k, v in OBS.items()}
CHK = pd.DataFrame(CHK)
summary = CHK.groupby('run', sort=False).agg({'obs in ObsFcstAna': 'sum', 'found offline': 'sum', 'built offline': 'sum',
                                             'position diff > 1e-4°': 'sum', 'max |F - fcst|': 'max',
                                             'median |F - fcst|': 'median'})
print('all 15 windows (5/1 03z-21z, 5/10 00z-21z), Metop-A/B/C')
print(summary.to_string(float_format=lambda v: f'{v:.1e}'))
print(f'\nbuilt in {time.time() - T_START:.0f} s')

**Takeaway:** every obs that the super-ob and thinning runs assimilated is reproduced offline at the same position
(and the raw-mode obs of the OL except one; in the OL-scaling super-ob rerun, one single-obs super-ob near 58.7N 97.6E
that sits almost exactly between two tile centres is assigned to the neighbouring tile offline), and the offline footprints applied to each run's own forecast reproduce
its ObsFcstAna `fcst` to ~1e-4 m³/m³. So the footprints and tile weights drawn below are the ones GEOSldas used.
("Built offline" is larger because the runs screen out obs on frozen or snow-covered tiles.)

### 1.1 The super-ob run scales its obs differently

The H SAF obs are scaled per tile to the model climatology, linear in the raw obs: `obs = a + b · raw`, and the
obs-error std is scaled by the same `b`, so `b = √obsvar / errstd` can be read from ObsFcstAna. Compared per
(window, satellite, tile) with the OL:

In [ ]:
def scaling(lab):
    m = OBS[lab]
    b = np.sqrt(m.obsvar) / m.sat.map(ERR[lab])
    return pd.DataFrame({'window': m.window, 'sat': m.sat, 'tile': m.tilenum, 'b': b, 'a': m.obs - b * m.sm})

SC = {lab: scaling(lab) for lab in RUNS}
fig, axs = plt.subplots(1, 2, figsize=(13, 4.2), constrained_layout=True)
rows = []
for lab in ['super-obs', 'super-obs, OL sc.', 'thin 25 km']:
    j = SC[lab].merge(SC['raw (OL)'], on=['window', 'sat', 'tile'], suffixes=('', '_ol'))
    r = j.b / j.b_ol
    for ax, day in zip(axs, DAYS):
        k = j.window.dt.date == day
        ax.hist(r[k], bins=np.linspace(0.5, 2.0, 151), histtype='step', lw=2, color=RCOL[lab],
                label=f'{lab}: {np.mean(np.abs(r[k] - 1) > 1e-3):.0%} differ')
        ax.set_title(f'{day:%Y-%m-%d}'); ax.set_xlabel('b (run) / b (OL), same window, satellite and tile')
        ax.set_yscale('log'); ax.legend()
    rows.append({'run': lab, 'tiles compared': len(j), 'median b / b_OL': r.median(),
                 '5th pct': r.quantile(0.05), '95th pct': r.quantile(0.95),
                 'differ by > 0.1%': f'{np.mean(np.abs(r - 1) > 1e-3):.1%}'})
plt.show()
print(pd.DataFrame(rows).set_index('run').to_string(float_format=lambda v: f'{v:.3f}'))

# the super-ob run's own (a, b) is a consistent per-tile linear map within a day
s = SC['super-obs'].copy()
s['sm'] = OBS['super-obs'].sm
s['obs'] = OBS['super-obs'].obs
s['day'] = s.window.dt.date
first = s.sort_values('window').drop_duplicates(['day', 'sat', 'tile']).set_index(['day', 'sat', 'tile'])
later = s.join(first[['a', 'b', 'window']], on=['day', 'sat', 'tile'], rsuffix='0')
later = later[later.window != later.window0]
e = np.abs(later.a0 + later.b0 * later.sm - later.obs)
print(f'\nsuper-obs: (a, b) of a tile from its first window predicts its later obs of the same day: '
      f'median |error| {e.median():.1e}, {np.mean(e > 1e-3):.1%} off by more than 1e-3 ({len(e):,} obs)')

The thinning run scales its obs exactly as the OL does (`b` equal on 98% of tiles; the rest are tiles whose raw-mode
obs mixes obs scaled differently). The super-ob run's `b` differs on almost every tile (median ratio ≈ 1.01, 5–95%
about 0.75–1.36), while within a day it is a consistent per-tile linear map (same-day check above).

This is deliberate: the thinning run uses the raw-obs scaling, which is correct for single raw obs, while the super-ob
run uses scaling statistics derived for the **old** super-ob approach (`e9c365d`), not for the new super-obs. What does
that do to the obs values? A super-ob is the mean of its raw obs, and the scaling is linear per tile, so scaling each raw
obs with the raw-obs (thinning) scaling of its tile and averaging gives the super-ob value the run would have had with
that scaling. Compared with the run's value:

In [ ]:
rows, X11 = [], []
for t in WINDOWS:
    raw = RAW[t][RAW[t].tile > 0]
    sc = raw.merge(SC['raw (OL)'], on=['window', 'sat', 'tile'], how='left') if 'window' in raw else \
        raw.assign(window=t).merge(SC['raw (OL)'], on=['window', 'sat', 'tile'], how='left')
    sc['sc'] = sc.a + sc.b * sc.sm
    lon = np.mod(sc.lon.to_numpy(np.float64) + 180.0, 360.0) - 180.0
    i = np.minimum(1440, ((lon + 180.0) / SUPEROB_GRID).astype(int) + 1)
    j = np.minimum(720, ((sc.lat.to_numpy(np.float64) + 90.0) / SUPEROB_GRID).astype(int) + 1)
    sc['cell'] = j * 1441 + i
    sc['lat64'], sc['lon64'] = sc.lat.astype(np.float64), lon
    # super-ob membership as in superobs(): all QC'd raw obs of the cell (with or without a tile) set the mean position;
    # here only cells whose raw obs all have a tile and a recovered scaling are used
    full = RAW[t].assign(lon64=np.mod(RAW[t].lon.to_numpy(np.float64) + 180.0, 360.0) - 180.0)
    fi = np.minimum(1440, ((full.lon64 + 180.0) / SUPEROB_GRID).astype(int) + 1)
    fj = np.minimum(720, ((full.lat.to_numpy(np.float64) + 90.0) / SUPEROB_GRID).astype(int) + 1)
    n_all = full.assign(cell=fj * 1441 + fi).groupby(['sat', 'cell']).size().rename('n_all')
    g = sc.groupby(['sat', 'cell']).agg(lat=('lat64', 'mean'), lon=('lon64', 'mean'), sc=('sc', 'mean'),
                                        ok=('sc', lambda x: np.isfinite(x).all()), n=('sc', 'size')).join(n_all)
    g = g[g.ok & (g.n == g.n_all)].reset_index()
    g['lat'], g['lon'] = g.lat.astype(np.float32), g.lon.astype(np.float32)
    so = OBS['super-obs'][OBS['super-obs'].window == t]
    x = so.merge(g[['sat', 'lat', 'lon', 'sc']], on=['sat', 'lat', 'lon'])
    X11.append(x.assign(d=x.obs - x.sc))
X11 = pd.concat(X11, ignore_index=True)
for day in DAYS:
    x = X11[X11.window.dt.date == day]
    rows.append({'day': f'{day:%m-%d}', 'super-obs compared': len(x), 'mean shift': x.d.mean(), 'std of shift': x.d.std(),
                 'O − F std, run scaling': (x.obs - x.fcst).std(), 'O − F std, raw-obs scaling': (x.sc - x.fcst).std()})
print('super-ob value with the run\'s (old super-ob) scaling minus the value with the raw-obs (thinning) scaling [m3/m3]')
print(pd.DataFrame(rows).set_index('day').to_string(float_format=lambda v: f'{v:.4f}' if abs(v) < 10 else f'{v:,.0f}'))

**Takeaway:** the old super-ob scaling is not biased for the new super-obs (mean shift ≈ −0.001 m³/m³), but it adds
about 0.015 m³/m³ of tile-to-tile noise, half the size of the whole O − F. The super-obs' O − F std is ≈ 0.030 with
it and ≈ 0.026 without (below the thinned and raw obs), and the run assimilates them with the same `errstd`. So the
super-ob run is handicapped by its scaling.

Hence the rerun **super-obs, OL sc.**: the same super-obs with the raw-obs (OL) scaling. Its `b` matches the OL's on
99.2% of tiles (table above), and its obs equal the OL scaling applied to the super-ob mean. Scaling the mean of the
raw obs on a tile with that tile's linear raw-obs map is nearly exact; averaging lowers the obs variance by only a few
percent, so a scaling derived for these super-obs would differ little. From here on, **super-obs, OL sc. vs thin 25
km** is the fair comparison; the old-scaling run is kept to show what the scaling did.

## 2. How many obs, how far apart

Per window, all satellites; nearest neighbours between obs of the same satellite.

In [ ]:
def nn_all(d):
    return np.concatenate([nn_km(g.lat.to_numpy(), g.lon.to_numpy()) for _, g in d.groupby(['window', 'sat'])])

rows = {}
for lab, d in OBS.items():
    nn = nn_all(d)
    for day in DAYS:
        k = d.window.dt.date == day
        nnk = nn_all(d[k])
        rows[lab, f'{day:%m-%d}'] = {'obs per window': k.sum() / d[k].window.nunique(),
                                     'raw obs per obs': d.n_raw[k].mean(),
                                     'NN spacing median [km]': np.nanmedian(nnk),
                                     'pairs < 20 km [% of obs]': 100 * np.mean(nnk < 20),
                                     'std(O − F)': d.OmF[k].std(), 'mean(O − F)': d.OmF[k].mean()}
COUNTS = pd.DataFrame(rows).T
print(COUNTS.to_string(float_format=lambda v: f'{v:,.3f}' if abs(v) < 1 else f'{v:,.1f}'))

fig, ax = plt.subplots(figsize=(11, 3.6), constrained_layout=True)
for lab, d in OBS.items():
    n = d.groupby('window').size()
    ax.plot(np.arange(len(n)), n.to_numpy(), 'o-', color=RCOL[lab], label=lab)
ax.set_xticks(np.arange(len(WINDOWS)), [t.strftime('%m-%d\n%Hz') for t in WINDOWS], fontsize=8)
ax.set_ylabel('H SAF obs in ObsFcstAna'); ax.legend(); ax.grid(alpha=0.3)
plt.show()

**Takeaway:** as offline, both reductions cut the H SAF obs well below raw mode (~80–86k per window): super-obs to
~29–32k, averaging ~3 raw obs each, thinning to ~21–23k single raw obs. Thinned obs are all ≥ 25 km apart (median
26.5 km); super-obs are 20 km apart at the median and half of them have a same-satellite neighbour closer than 20 km
(mean positions on either side of a cell edge). The two super-ob runs have the same obs (positions, counts); only the
scaling differs. With consistent scaling, super-obs have the smallest O − F std (0.026 and 0.024, against 0.029 and
0.027 for thinning): averaging removes a little noise. Each run's O − F is still against its own forecast, so this is
not a skill comparison (section 7).

## 3. The obs each run assimilated

SW Kansas, one window and satellite per day: the window with the most raw obs in the box. Obs coloured by O − F
(ObsFcstAna `obs − fcst`), raw QC'd obs as black dots, on the CF0360 tiles (light lines) and cube cells (dark lines).

In [ ]:
def best_window(day, box=REGION):
    best = max(((t, s) for t in WINDOWS if t.date() == day for s in SATS),
               key=lambda ts: len(inbox(RAW[ts[0]][RAW[ts[0]].sat == ts[1]], box)))
    return best

MAPS = {day: best_window(day) for day in DAYS}
print({f'{d}': f'{t:%H}z {SAT_LABEL[s]}' for d, (t, s) in MAPS.items()})
WIN = raster_window(REGION)
OMF_NORM = mcolors.TwoSlopeNorm(0, -0.08, 0.08)

fig, axs = plt.subplots(2, len(RUNS), figsize=(6.6 * len(RUNS), 11), constrained_layout=True)
for row, day in zip(axs, DAYS):
    t, sat = MAPS[day]
    raw_r = inbox(RAW[t][RAW[t].sat == sat], REGION)
    for ax, lab in zip(row, RUNS):
        d = inbox(OBS[lab][(OBS[lab].window == t) & (OBS[lab].sat == sat)], REGION)
        tile_background(ax, WIN)
        ax.plot(raw_r.lon, raw_r.lat, '.', color='k', ms=2.5, zorder=3)
        sc = ax.scatter(d.lon, d.lat, c=d.OmF, cmap='RdBu', norm=OMF_NORM, s=70 if lab != 'raw (OL)' else 30,
                        edgecolor='k', lw=0.5, zorder=4)
        finish(ax, REGION)
        ax.set_title(f'{lab}, {t:%Y-%m-%d %H}z {SAT_LABEL[sat]}: {len(d)} obs '
                     f'(from {int(d.n_raw.sum())} of {len(raw_r)} raw obs)', fontsize=10)
fig.colorbar(sc, ax=axs, shrink=0.6, label='O − F [m³/m³] (own forecast, own scaling)')
fig.savefig(FIG_DIR / 'obs_maps_kansas.png', dpi=150)
plt.show()

The raw obs sit on the fixed H SAF Fibonacci grid, so a full Metop-A pass over the box hits the same grid points on
both days; the obs **positions** in the two rows are nearly the same, the O − F is not. On 5/1 03z all three runs
share the forecast, and the obs are much drier than it; by 5/10 both DA runs have dried the model and their O − F
is close to zero or positive, while the OL's is still negative.

**Takeaway:** raw mode assimilates (here: monitors) nearly every raw obs, one per tile. Thinning keeps a sparse,
fairly even subset of single raw obs at their own positions, ≥ 25 km apart. Super-obs sit at the mean position of the
raw obs in each 0.25° cell, so they are spaced by the grid but shift towards wherever the raw obs of the cell are; at
swath edges and in partial cells they can come close to each other.

## 4. Footprints, and how often each tile is used

The same windows. Outlines are the support of each footprint: the 25 km cut-off circle of R (raw, thinning) and the
q = 4 (2σ) ellipse of N (super-obs). Tiles are coloured by the number of footprints that contain them.

In [ ]:
def weights(lab, d):
    if is_so(lab):
        return fov_N(d.lat, d.lon, d[['Sxx', 'Syy', 'Sxy']].to_numpy())
    return fov_R(d.lat, d.lon)

CNT_NORM = mcolors.BoundaryNorm(np.arange(0.5, 13.5), 256)
fig, axs = plt.subplots(2, len(RUNS), figsize=(6.6 * len(RUNS), 11), constrained_layout=True)
pad = (0.3, 0.3, 0.4, 0.4)
ext = (REGION[0] - pad[0], REGION[1] + pad[1], REGION[2] - pad[2], REGION[3] + pad[3])
inner = np.flatnonzero((COM_LAT > REGION[0]) & (COM_LAT < REGION[1]) & (COM_LON > REGION[2]) & (COM_LON < REGION[3]))
FOOT = []
for row, day in zip(axs, DAYS):
    t, sat = MAPS[day]
    for ax, lab in zip(row, RUNS):
        d = inbox(OBS[lab][(OBS[lab].window == t) & (OBS[lab].sat == sat)], ext)
        W = weights(lab, d)
        cnt = np.asarray((W > 0).sum(0)).ravel()
        used = np.flatnonzero(cnt)
        tile_background(ax, WIN)
        pm = paint_tiles(ax, WIN, used, cnt[used].astype(float), cmap='magma_r', norm=CNT_NORM)
        kind = 'N' if is_so(lab) else 'R'
        for _, r in d.iterrows():
            x, y = footprint_outline(r, kind)
            ax.plot(x, y, color=RCOL[lab] if lab != 'raw (OL)' else 'tab:blue', lw=0.7, zorder=3)
        ax.plot(d.lon, d.lat, 'o', color='w', ms=3, mec='k', mew=0.5, zorder=4)
        finish(ax, REGION)
        ci = cnt[inner]
        FOOT.append({'day': f'{day:%m-%d}', 'run': lab, 'obs': len(d), 'tiles unused [%]': 100 * np.mean(ci == 0),
                     'footprints per used tile (median)': np.median(ci[ci > 0]),
                     'max': ci.max()})
        ax.set_title(f'{lab}, {t:%m-%d %H}z: {np.mean(ci == 0):.0%} of tiles unused, '
                     f'median {np.median(ci[ci > 0]):.0f} footprints per used tile', fontsize=10)
fig.colorbar(pm, ax=axs, shrink=0.6, ticks=np.arange(1, 13), label='footprints containing the tile (one satellite)')
fig.savefig(FIG_DIR / 'footprints_kansas.png', dpi=150)
plt.show()
print(pd.DataFrame(FOOT).set_index(['day', 'run']).to_string(float_format=lambda v: f'{v:.1f}'))

**Takeaway:** raw-mode footprints (25 km cut-off, obs 12.5 km apart) pile up: the median tile is in 8 of them (up
to 13), for one satellite in one window. Super-obs bring that to 4, thinning to 2, and both still leave almost no
tile unused (≤ 3%): 25 km cut-off circles around obs ≥ 25 km apart overlap their neighbours, and super-ob ellipses are
at least as large as the raw circle.

## 5. Forecast weights on tiles

### 5.1 One super-ob and the thinned obs around it

A super-ob built from 4+ raw obs near the middle of the box (5/1 window). Left: the raw-mode obs of the same raw
obs, footprints combined (each obs's normalised weights summed, divided by the number of obs). Middle: the super-ob's
normalised weights. Right: the thinned obs whose footprints overlap the super-ob's, combined the same way. Tile
colour = normalised weight ŵ (sums to 1 over each panel).

In [ ]:
t, sat = MAPS[DAYS[0]]
so_r = inbox(OBS['super-obs'][(OBS['super-obs'].window == t) & (OBS['super-obs'].sat == sat)], REGION)
cen = np.array([np.mean(REGION[:2]), np.mean(REGION[2:])])
so_r = so_r[so_r.n_raw >= 4]
s0 = so_r.iloc[np.argmin(np.hypot(so_r.lat - cen[0], (so_r.lon - cen[1]) * np.cos(np.radians(cen[0]))))]
cell_lat0 = np.floor((s0.lat + 90) / SUPEROB_GRID) * SUPEROB_GRID - 90
cell_lon0 = np.floor((s0.lon + 180) / SUPEROB_GRID) * SUPEROB_GRID - 180
raw_c = RAW[t][(RAW[t].sat == sat) & RAW[t].lat.between(cell_lat0, cell_lat0 + SUPEROB_GRID)
               & RAW[t].lon.between(cell_lon0, cell_lon0 + SUPEROB_GRID)]
rm_c = OBS['raw (OL)'][(OBS['raw (OL)'].window == t) & (OBS['raw (OL)'].sat == sat)
                       & OBS['raw (OL)'].tilenum.isin(raw_c.tile)]
W0 = weights('super-obs', s0.to_frame().T.astype({'lat': float, 'lon': float, 'Sxx': float, 'Syy': float, 'Sxy': float}))
th_all = OBS['thin 25 km'][(OBS['thin 25 km'].window == t) & (OBS['thin 25 km'].sat == sat)]
th_all = inbox(th_all, (s0.lat - 0.6, s0.lat + 0.6, s0.lon - 0.8, s0.lon + 0.8))
Wt = weights('thin 25 km', th_all)
th_c = th_all[np.asarray((Wt[:, W0.indices] > 0).sum(1)).ravel() > 0]
n_in_cell = int((th_c.lat.between(cell_lat0, cell_lat0 + SUPEROB_GRID)
                 & th_c.lon.between(cell_lon0, cell_lon0 + SUPEROB_GRID)).sum())

def combined(W):
    w = np.asarray(W.sum(0)).ravel() / W.shape[0]
    k = np.flatnonzero(w)
    return k, w[k]

panels = [('raw (OL)', f'{len(rm_c)} raw-mode obs, combined', combined(weights('raw (OL)', rm_c)), rm_c),
          ('super-obs', f'1 super-ob ({int(s0.n_raw)} raw obs), C = 12.5²I + S', (W0.indices, W0.data), None),
          ('thin 25 km', f'{n_in_cell} thinned obs in the cell; {len(th_c)} whose footprints overlap the super-ob\'s, combined', combined(weights('thin 25 km', th_c)), th_c)]
pad = 0.4
box0 = (s0.lat - pad, s0.lat + pad, s0.lon - pad / np.cos(np.radians(s0.lat)), s0.lon + pad / np.cos(np.radians(s0.lat)))
win0 = raster_window(box0)
vmax = max(p[2][1].max() for p in panels)
fig, axs = plt.subplots(1, 3, figsize=(20, 6.8), constrained_layout=True)
for ax, (lab, title, (tl, wv), dd) in zip(axs, panels):
    tile_background(ax, win0)
    pm = paint_tiles(ax, win0, tl, wv, cmap='YlOrRd', vmin=0, vmax=vmax)
    if dd is not None:
        for _, r in dd.iterrows():
            x, y = footprint_outline(r, 'R')
            ax.plot(x, y, color='tab:blue' if lab == 'raw (OL)' else RCOL[lab], lw=1)
        ax.plot(dd.lon, dd.lat, 'o', color='w', ms=6, mec='k', zorder=5)
    else:
        for nsig, lw in [(1, 2.2), (2, 1.3)]:
            C2 = FOV_RAW_KM ** 2 * np.eye(2) + np.array([[s0.Sxx, s0.Sxy], [s0.Sxy, s0.Syy]])
            e = ellipse_xy(C2, nsig)
            ax.plot(s0.lon + e[:, 0] * KM2DEG / np.cos(np.radians(s0.lat)), s0.lat + e[:, 1] * KM2DEG,
                    color=RCOL['super-obs'], lw=lw)
        ax.plot(s0.lon, s0.lat, '*', color='w', ms=16, mec='k', zorder=6)
    ax.plot(raw_c.lon, raw_c.lat, 'k.', ms=8, zorder=5)
    ax.add_patch(Rectangle((cell_lon0, cell_lat0), SUPEROB_GRID, SUPEROB_GRID, fill=False, ec='tab:red', lw=1.3, ls='--'))
    finish(ax, box0)
    ax.set_title(f'{title}\n{len(tl)} tiles, {1 / np.sum(wv ** 2):.1f} effective', fontsize=10)
fig.colorbar(pm, ax=axs, shrink=0.8, label='ŵ (normalised tile weight)')
fig.suptitle(f'{SAT_LABEL[sat]}, {t:%Y-%m-%d %H}z: super-ob at {s0.lat:.2f}N {s0.lon:.2f}E; '
             f'black dots = raw obs in its 0.25° cell (red)', fontsize=12)
fig.savefig(FIG_DIR / 'weights_one_superob.png', dpi=150)
plt.show()

**Takeaway:** the super-ob puts its weight on about the same tiles as the raw-mode obs it replaces (18 vs 22 tiles,
8 vs 10 effective), with a similar shape, so it predicts what those raw obs would on average. With thinning, none of
the cell's 5 raw obs survives here: the greedy pass kept neighbours outside the cell, and this area is seen only
through the tails of 7 footprints centred up to ~25 km away. Which raw obs survive depends on the visiting order,
not on the data.

### 5.2 Total weight per tile

Summed over all obs of the window and satellite, Σ ŵ per tile, divided by the tile area: how much prediction
weight each part of the box carries, in obs-equivalents per 100 km². (Σ ŵ itself grows with tile area, since the
footprint weights are area-weighted.) Over the box, Σ ŵ is about the number of obs.

In [ ]:
SUMW_NORM = mcolors.LogNorm(0.04, 0.8)
fig, axs = plt.subplots(2, len(RUNS), figsize=(6.6 * len(RUNS), 11), constrained_layout=True)
SW = []
for row, day in zip(axs, DAYS):
    t, sat = MAPS[day]
    for ax, lab in zip(row, RUNS):
        d = inbox(OBS[lab][(OBS[lab].window == t) & (OBS[lab].sat == sat)], ext)
        sw = np.asarray(weights(lab, d).sum(0)).ravel()
        used = np.flatnonzero(sw)
        tile_background(ax, WIN)
        dens = 100 * sw / AREA                                         # obs-equivalents per 100 km2
        pm = paint_tiles(ax, WIN, used, dens[used], cmap='viridis', norm=SUMW_NORM)
        ax.plot(d.lon, d.lat, '.', color='w', ms=3, mec='k', mew=0.3, zorder=4)
        finish(ax, REGION)
        si, di, ai = sw[inner], dens[inner], AREA[inner]
        mu = np.average(di, weights=ai)
        cv = np.sqrt(np.average((di - mu) ** 2, weights=ai)) / mu       # area-weighted, over all tiles in the box
        SW.append({'day': f'{day:%m-%d}', 'run': lab, 'Σŵ in box': si.sum(),
                   'mean per 100 km²': mu, 'area-weighted CV': cv})
        ax.set_title(f'{lab}, {t:%m-%d %H}z: Σŵ over the box {si.sum():.0f}; per 100 km²: mean {mu:.2f}, CV {cv:.2f}',
                     fontsize=10)
fig.colorbar(pm, ax=axs, shrink=0.6, label='Σ ŵ per 100 km² (all obs of the window and satellite)')
fig.savefig(FIG_DIR / 'sum_weights_kansas.png', dpi=150)
plt.show()
print(pd.DataFrame(SW).set_index(['day', 'run']).to_string(float_format=lambda v: f'{v:.2f}'))

**Takeaway:** per unit area, raw mode puts 0.40 obs-equivalents per 100 km² on the box, super-obs 0.15 and thinning
0.10 (one satellite, one window). Super-obs also spread it most evenly (area-weighted CV 0.28 vs 0.31 for thinning and
0.38 for raw mode, whose coverage follows the swath edge and the screening): their ellipses tile the 0.25° grid,
while thinned circles overlap in some places and leave thinner coverage in others.

### 5.3 Where the obs fall, pass after pass

The raw obs sit on the fixed H SAF grid, so the same grid points come back on every pass over a place. Thinning keeps a
subset that depends on the visiting order (file order, which starts wherever the swath and the QC'd obs start), so the
kept set can change from pass to pass even when the raw obs are the same. A super-ob is the mean position of the raw
obs in its cell, so it moves only when the set of raw obs in the cell changes. Here, from the offline obs of section 1
(before the model-side screening), for all 15 windows and three satellites.

First, one pass that repeats exactly: Metop-A 03z over SW Kansas on 5/1 and 5/10, with the same raw obs.

In [ ]:
t1, t2, sat = dt.datetime(2019, 5, 1, 3), dt.datetime(2019, 5, 10, 3), 'metop_a'
r1, r2 = inbox(RAW[t1][(RAW[t1].sat == sat) & (RAW[t1].tile > 0)], REGION), inbox(RAW[t2][(RAW[t2].sat == sat) & (RAW[t2].tile > 0)], REGION)
same_raw = set(zip(r1.lat, r1.lon)) == set(zip(r2.lat, r2.lon))
k1, k2 = set(zip(r1.lat[r1.kept], r1.lon[r1.kept])), set(zip(r2.lat[r2.kept], r2.lon[r2.kept]))
s1 = inbox(SOF[t1][(SOF[t1].sat == sat) & (SOF[t1].tile > 0)], REGION)
s2 = inbox(SOF[t2][(SOF[t2].sat == sat) & (SOF[t2].tile > 0)], REGION)
p1, p2 = set(zip(s1.lat, s1.lon)), set(zip(s2.lat, s2.lon))
print(f'raw obs in the box: {len(r1)} and {len(r2)}, identical positions: {same_raw}')
print(f'thinning kept {len(k1)} and {len(k2)}; at the same grid point on both days: {len(k1 & k2)}')
print(f'super-obs: {len(p1)} and {len(p2)}; at the same position on both days: {len(p1 & p2)}')

fig, axs = plt.subplots(1, 2, figsize=(19, 8.5), constrained_layout=True)
for ax, lab in zip(axs, ['thin 25 km', 'super-obs']):
    tile_background(ax, WIN, edge_lw=0.3)
    ax.plot(r1.lon, r1.lat, '.', color='0.45', ms=3, zorder=3)
    if lab == 'thin 25 km':
        a, b = np.array(sorted(k1)), np.array(sorted(k2))
    else:
        a, b = np.array(sorted(p1)), np.array(sorted(p2))
        for x in np.arange(np.ceil(REGION[2] / SUPEROB_GRID) * SUPEROB_GRID, REGION[3], SUPEROB_GRID):
            ax.axvline(x, color='tab:red', lw=0.6, ls='--', zorder=2)
        for y in np.arange(np.ceil(REGION[0] / SUPEROB_GRID) * SUPEROB_GRID, REGION[1], SUPEROB_GRID):
            ax.axhline(y, color='tab:red', lw=0.6, ls='--', zorder=2)
    ax.scatter(a[:, 1], a[:, 0], s=110, facecolor='none', edgecolor=RCOL[lab], lw=2, zorder=4, label=f'{t1:%m-%d %H}z')
    ax.scatter(b[:, 1], b[:, 0], s=45, marker='x', color='k', lw=1.5, zorder=5, label=f'{t2:%m-%d %H}z')
    n_same = len(k1 & k2) if lab == 'thin 25 km' else len(p1 & p2)
    ax.set_title(f'{lab}: {len(a)} and {len(b)} obs, {n_same} at the same place on both days', fontsize=11)
    ax.legend(loc='upper right', fontsize=9)
    finish(ax, REGION)
fig.suptitle(f'{SAT_LABEL[sat]} 03z over SW Kansas on two days with the same raw obs (grey dots)', fontsize=13)
fig.savefig(FIG_DIR / 'locations_same_pass_two_days.png', dpi=150)
plt.show()

Then all passes: how often each grid point is kept by thinning, and where the super-obs sit, in every window and
satellite (left: every H121 grid point in the box, coloured by the fraction of the passes that observed it in which it
was kept, sized by the number of those passes; right: super-ob positions of all passes, coloured by the number of raw
obs averaged). Statistics are global, except the tile weights (central US, 33–43N, 105–90W).

In [ ]:
def point_key(lat, lon):
    return (np.round(np.asarray(lat, np.float64) * 1e4).astype(np.int64) + 900000) * 4000000 + \
           (np.round(np.asarray(lon, np.float64) * 1e4).astype(np.int64) + 1800000)

RA = pd.concat([RAW[t][RAW[t].tile > 0].assign(window=t) for t in WINDOWS], ignore_index=True)
RA['pid'] = point_key(RA.lat, RA.lon)
SA = pd.concat([SOF[t][SOF[t].tile > 0].assign(window=t) for t in WINDOWS], ignore_index=True)
base = RA.kept.mean()

# thinning: fraction of passes in which a grid point is kept
GP = RA.groupby('pid').agg(n=('kept', 'size'), k=('kept', 'sum'), lat=('lat', 'first'), lon=('lon', 'first'))
GP['p'] = GP.k / GP.n
g8 = GP[GP.n >= 8]
rng = np.random.default_rng(0)
sim = rng.binomial(g8.n, base) / g8.n
pr = RA[['pid', 'sat', 'window', 'kept']]
pr = pr.merge(pr, on=['pid', 'sat'])
pr = pr[pr.window_x < pr.window_y]
p_again = pr.kept_y[pr.kept_x].mean()

# super-obs: offset from the cell centre, and scatter of the position per cell across passes
SA['clat0'] = np.floor((SA.lat + 90) / SUPEROB_GRID) * SUPEROB_GRID - 90
SA['clon0'] = np.floor((SA.lon + 180) / SUPEROB_GRID) * SUPEROB_GRID - 180
SA['x_km'] = (SA.lon - SA.clon0 - SUPEROB_GRID / 2) * KM_PER_DEG * np.cos(np.radians(SA.lat))
SA['y_km'] = (SA.lat - SA.clat0 - SUPEROB_GRID / 2) * KM_PER_DEG
SA['r_km'] = np.hypot(SA.x_km, SA.y_km)
CS = SA.groupby(['clat0', 'clon0']).agg(n=('x_km', 'size'), sx=('x_km', 'std'), sy=('y_km', 'std'))
CS = CS[CS.n >= 4]
CS['s_km'] = np.hypot(CS.sx, CS.sy)

def jump_km(df):
    """per obs: distance to the nearest obs of the previous pass of the same satellite that covered the same place"""
    out = []
    for sat in SATS:
        w = sorted(df[df.sat == sat].window.unique())
        for t0, t1 in zip(w[:-1], w[1:]):
            a, b = df[(df.sat == sat) & (df.window == t0)], df[(df.sat == sat) & (df.window == t1)]
            rr = RA[(RA.sat == sat) & (RA.window == t0)]
            if len(a) < 2 or len(b) < 2 or len(rr) < 2:
                continue
            d = 2 * np.arcsin(cKDTree(to_xyz(a.lat, a.lon)).query(to_xyz(b.lat, b.lon))[0] / 2) * MAPL_RADIUS_KM
            dr = 2 * np.arcsin(cKDTree(to_xyz(rr.lat, rr.lon)).query(to_xyz(b.lat, b.lon))[0] / 2) * MAPL_RADIUS_KM
            out.append(d[dr < 1.0])                 # the earlier pass had a raw obs at this place
    return np.concatenate(out)

J_thin, J_so = jump_km(RA[RA.kept]), jump_km(SA)

# tile weights across passes (central US), for tiles fully inside the swath (raw-mode weight >= 80% of its median)
BOX_US = (33.0, 43.0, -105.0, -90.0)
tiles_us = np.flatnonzero((COM_LAT > BOX_US[0]) & (COM_LAT < BOX_US[1]) & (COM_LON > BOX_US[2]) & (COM_LON < BOX_US[3]))
ext_us = (BOX_US[0] - 0.5, BOX_US[1] + 0.5, BOX_US[2] - 0.6, BOX_US[3] + 0.6)
SWk = {k: [] for k in ['raw', 'super-obs', 'thin 25 km']}
for t in WINDOWS:
    for sat in SATS:
        r = inbox(RA[(RA.window == t) & (RA.sat == sat)], ext_us)
        if len(r) < 50:
            continue
        rm, th = tile_average(r), tile_average(r[r.kept])
        so = inbox(SA[(SA.window == t) & (SA.sat == sat)], ext_us)
        col = lambda W: np.asarray(W.sum(0)).ravel()[tiles_us]
        SWk['raw'].append(col(fov_R(rm.lat, rm.lon)))
        SWk['thin 25 km'].append(col(fov_R(th.lat, th.lon)))
        SWk['super-obs'].append(col(fov_N(so.lat, so.lon, so[['Sxx', 'Syy', 'Sxy']].to_numpy())))
SWk = {k: np.array(v) for k, v in SWk.items()}
with warnings.catch_warnings(), np.errstate(invalid='ignore'):      # tiles never seen give all-NaN columns
    warnings.simplefilter('ignore', RuntimeWarning)
    inside = SWk['raw'] >= 0.8 * np.nanmedian(np.where(SWk['raw'] > 0, SWk['raw'], np.nan), axis=0)[None, :]
    enough = inside.sum(0) >= 4
    CV = {}
    for k, S in SWk.items():
        m = np.where(inside, S, np.nan)
        CV[k] = (np.nanstd(m, 0) / np.nanmean(m, 0))[enough]

rows = {'thin 25 km': {'location repeats previous pass [%]': 100 * np.mean(J_thin < 0.5),
                       'median move from previous pass [km]': np.median(J_thin),
                       'CV of tile weight across passes (median)': np.nanmedian(CV['thin 25 km'])},
        'super-obs': {'location repeats previous pass [%]': 100 * np.mean(J_so < 0.5),
                      'median move from previous pass [km]': np.median(J_so),
                      'CV of tile weight across passes (median)': np.nanmedian(CV['super-obs'])}}
for k in ['super-obs', 'thin 25 km']:
    rel = (SWk[k] / np.where(SWk['raw'] > 0, SWk['raw'], np.nan))[inside]
    rows[k]['tile weight / raw mode (median)'] = np.nanmedian(rel)
    rows[k]['tile-passes with < 10% of raw weight [%]'] = 100 * np.nanmean(rel < 0.1)
print(pd.DataFrame(rows).T.to_string(float_format=lambda v: f'{v:.2f}'))
print(f'\nthinning: {base:.1%} of raw obs kept; P(kept again | kept, same point and satellite, another window) = {p_again:.3f}; '
      f'grid points seen >= 8 times: {np.mean(g8.k == 0):.0%} never kept, spread of the kept fraction {g8.p.std():.2f} '
      f'(random selection {sim.std():.2f}, same points every pass {np.sqrt(base * (1 - base)):.2f})')
print(f'super-obs: offset from the cell centre median {SA.r_km.median():.1f} km (single-obs super-obs '
      f'{SA.r_km[SA.n_raw == 1].median():.1f} km); scatter per cell across passes median {CS.s_km.median():.1f} km, '
      f'95th pct {CS.s_km.quantile(0.95):.1f} km')

fig, axs = plt.subplots(1, 2, figsize=(19, 8.5), constrained_layout=True)
gk = inbox(GP.reset_index(), REGION)
tile_background(axs[0], WIN, edge_lw=0.3)
sc_ = axs[0].scatter(gk.lon, gk.lat, c=gk.p, cmap='viridis', vmin=0, vmax=1, s=8 + 5 * gk.n, edgecolor='k', lw=0.3, zorder=4)
fig.colorbar(sc_, ax=axs[0], shrink=0.7, label='fraction of passes in which the grid point is kept')
axs[0].set_title(f'thinning: each H121 grid point, coloured by how often it is kept ({len(WINDOWS)} windows, 3 satellites)',
                 fontsize=11)
sk = inbox(SA, REGION)
tile_background(axs[1], WIN, edge_lw=0.3)
axs[1].plot(gk.lon, gk.lat, '.', color='0.6', ms=2, zorder=3)
for sat_, mk in zip(SATS, ['o', 's', '^']):
    k = sk.sat == sat_
    sc2 = axs[1].scatter(sk.lon[k], sk.lat[k], c=sk.n_raw[k], cmap='plasma', vmin=1, vmax=8, s=22, marker=mk,
                         edgecolor='k', lw=0.2, zorder=4, label=SAT_LABEL[sat_])
fig.colorbar(sc2, ax=axs[1], shrink=0.7, label='raw obs averaged into the super-ob')
axs[1].legend(loc='upper right', fontsize=9)
axs[1].set_title('super-obs: positions in every pass (grey = grid points); most cells show one position', fontsize=11)
for ax in axs:
    for x in np.arange(np.ceil(REGION[2] / SUPEROB_GRID) * SUPEROB_GRID, REGION[3], SUPEROB_GRID):
        ax.axvline(x, color='tab:red', lw=0.5, ls='--', zorder=2)
    for y in np.arange(np.ceil(REGION[0] / SUPEROB_GRID) * SUPEROB_GRID, REGION[1], SUPEROB_GRID):
        ax.axhline(y, color='tab:red', lw=0.5, ls='--', zorder=2)
    finish(ax, REGION)
fig.savefig(FIG_DIR / 'locations_all_passes_kansas.png', dpi=150)
plt.show()

fig, axs = plt.subplots(1, 3, figsize=(19, 4.8), constrained_layout=True)
bins = np.linspace(0, 1, 21)
axs[0].hist(g8.p, bins=bins, color=RCOL['thin 25 km'], alpha=0.75, label='thinning')
axs[0].hist(sim, bins=bins, histtype='step', color='k', lw=1.5, label=f'random selection ({base:.0%})')
axs[0].set_xlabel('fraction of passes in which a grid point is kept'); axs[0].set_ylabel('grid points seen ≥ 8 times')
axs[0].legend(); axs[0].set_title('thinning: the kept points change from pass to pass')
bins = np.linspace(0, 30, 61)
axs[1].hist(J_thin, bins=bins, histtype='step', lw=2, color=RCOL['thin 25 km'], density=True, label='thinning')
axs[1].hist(J_so, bins=bins, histtype='step', lw=2, color=RCOL['super-obs'], density=True, label='super-obs')
axs[1].set_yscale('log'); axs[1].set_xlabel('distance to the nearest obs of the previous pass over the same place [km]')
axs[1].legend(); axs[1].set_title('how far the obs locations move between passes')
bins = np.linspace(0, 1.0, 51)
for k, col in [('super-obs', RCOL['super-obs']), ('thin 25 km', RCOL['thin 25 km'])]:
    axs[2].hist(CV[k], bins=bins, histtype='step', lw=2, color=col, density=True, label=f'{k}: median {np.nanmedian(CV[k]):.2f}')
axs[2].set_xlabel('CV across passes of the weight a tile gets (tile inside the swath)'); axs[2].legend()
axs[2].set_title('how steady the weight on each tile is (central US)')
fig.savefig(FIG_DIR / 'locations_stats.png', dpi=150)
plt.show()

**Takeaway:** thinning keeps a different subset of grid points on every pass. Even on two passes with identical raw
obs, the kept sets differ, because the greedy chain depends on where the swath and its QC'd obs start. Globally, only
about a third of thinned obs sit where an obs was kept on the previous pass over the same place, a kept point is only
a little more likely than average to be kept again, and about a fifth of the grid points are never kept (they always
lose to a neighbour). So the weight each tile gets changes from pass to pass (CV ≈ 0.17), and about 1 in 20
tile-passes inside the swath is nearly unobserved, in places that move. Super-obs are offset from their cell centres
(more so with few raw obs), but in a fully covered cell the same grid points fall in the cell every pass, so the
super-ob is at the same place and every tile gets the same weight; they move only at swath edges and in partial cells.

For the DA, thinning works like a fresh random subsample each cycle: over many cycles every grid point is used, but
each cycle's increments carry sampling noise at the ~25 km scale (the less-dry thinned subset over Kansas on 5/1 03z
in section 6 is one example). Super-obs are a fixed operator, which suits per-tile climatology scaling, but anything
tied to location persists, including super-obs of different Metop satellites at identical positions (which, with
consistent scaling, does not make the analysis overshoot: section 6.1).

## 6. What the analysis did: increments

SFMC analysis increment (`SFMC_ANA − SFMC_FCST`, ensemble mean) for the same windows. Increments come from all three
Metop satellites in the window, not just the one mapped above. The output has a noise floor: even the OL, which
assimilates nothing, has |ANA − FCST| up to 1e-3 m³/m³ (ensemble perturbations), so tiles count as updated only above
that.

In [ ]:
INC_NORM = mcolors.TwoSlopeNorm(0, -0.03, 0.03)
DA_RUNS = [lab for lab in RUNS if lab != 'raw (OL)']
fig, axs = plt.subplots(2, len(DA_RUNS), figsize=(6.6 * len(DA_RUNS), 11), constrained_layout=True)
INC = []
for row, day in zip(axs, DAYS):
    t, sat = MAPS[day]
    for ax, lab in zip(row, DA_RUNS):
        inc = ANA[lab, t] - FCST[lab, t]
        tile_background(ax, WIN)
        near = np.flatnonzero((COM_LAT > ext[0]) & (COM_LAT < ext[1]) & (COM_LON > ext[2]) & (COM_LON < ext[3]))
        pm = paint_tiles(ax, WIN, near, inc[near], cmap='BrBG', norm=INC_NORM)
        d = inbox(OBS[lab][OBS[lab].window == t], REGION)
        ax.plot(d.lon, d.lat, '.', color='k', ms=2.5, zorder=4)
        finish(ax, REGION)
        ax.set_title(f'{lab}, {t:%Y-%m-%d %H}z: {len(d)} obs (all satellites)', fontsize=10)
fig.colorbar(pm, ax=axs, shrink=0.6, label='SFMC increment [m³/m³]')
fig.savefig(FIG_DIR / 'increments_kansas.png', dpi=150)
plt.show()

INC_FLOOR = 1e-3
for lab in RUNS:
    for day in DAYS:
        incs = [ANA[lab, t] - FCST[lab, t] for t in WINDOWS if t.date() == day]
        a = np.concatenate([x[np.abs(x) > INC_FLOOR] for x in incs])
        INC.append({'run': lab, 'day': f'{day:%m-%d}', 'tiles updated per window': a.size / len(incs),
                    'rms increment': np.sqrt(np.mean(a ** 2)) if a.size else np.nan,
                    'mean increment': a.mean() if a.size else np.nan,
                    '|incr| > 0.02 [% of updated]': 100 * np.mean(np.abs(a) > 0.02) if a.size else np.nan})
print(f'global, all windows of the day, tiles with |increment| > {INC_FLOOR:g} only')
print(pd.DataFrame(INC).set_index(['run', 'day']).to_string(float_format=lambda v: f'{v:,.4f}' if abs(v) < 10 else f'{v:,.0f}'))

**Takeaway:** on 5/1 03z (same forecast, same raw obs) the old-scaling super-ob run dries most of the box by
≥ 0.03 m³/m³ in one coherent patch. With the OL scaling the same super-obs give moderate increments of both signs,
comparable to thinning's, and by 5/10 the smoothest field of the three. Globally (all windows), the OL-scaled super-ob
run has the **smallest** increments: rms 0.013 and 0.011 m³/m³ on 5/1 and 5/10, against 0.015 and 0.013 for thinning
and 0.018 and 0.014 for the old scaling, and the fewest above 0.02 (10% and 6% of updates). Same `errstd`, smaller
innovations (O − F std 0.026 vs 0.029, section 2), so smaller increments.

### 6.1 Does the analysis overshoot its obs?

In obs space, ObsFcstAna gives the innovation O − F and the analysis increment A − F at each assimilated obs. With
correct error statistics A − F lies between 0 and O − F; |A − F| > |O − F| means the analysis moved further than the
obs asked for, a sign that correlated obs are counted as independent. Per run, for SW Kansas on the shared-forecast
window (5/1 03z) and globally per day; and how similar the runs' tile increments are on 5/1 03z, where only the obs differ.

In [ ]:
OV = []
for lab in DA_RUNS:
    for name, box, sel in [('SW Kansas, 05-01 03z', REGION, lambda w: w == WINDOWS[0]),
                           ('global, 05-01', (-90, 90, -180, 180), lambda w: w.dt.date == DAYS[0]),
                           ('global, 05-10', (-90, 90, -180, 180), lambda w: w.dt.date == DAYS[1])]:
        o = inbox(OBS[lab][sel(OBS[lab].window)], box)
        omf, amf = o.obs - o.fcst, o.ana - o.fcst
        OV.append({'where': name, 'run': lab, 'obs': len(o), 'mean O − F': omf.mean(), 'mean A − F': amf.mean(),
                   'slope of A − F on O − F': np.polyfit(omf, amf, 1)[0],
                   '|A − F| > |O − F| [%]': 100 * np.mean(np.abs(amf) > np.abs(omf))})
print(pd.DataFrame(OV).set_index(['where', 'run']).to_string(float_format=lambda v: f'{v:+.4f}' if abs(v) < 1 else f'{v:,.2f}'))

t0 = WINDOWS[0]
inc0 = {lab: ANA[lab, t0] - FCST[lab, t0] for lab in DA_RUNS}
upd = np.any([np.abs(v) > INC_FLOOR for v in inc0.values()], axis=0)
C = pd.DataFrame([[np.corrcoef(inc0[a][upd], inc0[b][upd])[0, 1] for b in DA_RUNS] for a in DA_RUNS],
                 index=DA_RUNS, columns=DA_RUNS)
print(f'\ncorrelation of SFMC increments on 05-01 03z (same forecast), {upd.sum():,} tiles updated by any run')
print(C.to_string(float_format=lambda v: f'{v:.2f}'))
print('rms increment on those tiles: ' + ', '.join(f'{lab} {np.sqrt(np.mean(inc0[lab][upd] ** 2)):.4f}' for lab in DA_RUNS))

**Takeaway (6.1):** the Kansas overshoot was the scaling. With the old scaling, the super-ob analysis moved 2.3×
further than its obs on average (A − F −0.037 vs O − F −0.016) and past the obs at 58% of them. With the OL scaling
the same super-obs give A − F −0.011 for O − F −0.011, and 28% past the obs (thinning 20%). Globally all three runs
take a similar fraction of the innovation (slope 0.33–0.35 on 5/1, 0.21–0.26 on 5/10) and overshoot at 18–23% of obs.
Stacked super-obs of different satellites at identical positions do not, on this evidence, make the EnKF overcount.
On 5/1 03z, where only the obs differ, the OL-scaled super-ob increments correlate 0.66 with thinning's (old scaling:
0.47), so about half of what looked like a method difference was the scaling.

## 7. Each run's forecast against common obs

Each run's own O − F uses its own obs set and scaling, so it cannot rank the runs. Here every run's **forecast** is
compared with the **same** obs. On 5/1 03z all runs share the forecast (bit-identical `SFMC_FCST`); after that, each
forecast carries its own DA history.

### 7.1 The raw H SAF obs

The OL's raw-mode obs (OL scaling), predicted from each run's `SFMC_FCST` with the raw-obs footprint R at the obs
position, on the cube cells that passed screening in all runs. **Not independent, and not neutral:** the
thinning run assimilated a subset of exactly these obs in exactly this scaling; the super-ob run assimilated
averages of them in a different scaling (section 1.1). So this favours thinning by construction.

In [ ]:
ref = OBS['raw (OL)']
common = None
for lab in RUNS:
    k = OBS[lab][['window', 'tilenum']].drop_duplicates()
    k = set(zip(k.window, CELL[k.tilenum.to_numpy() - 1]))
    common = k if common is None else common & k
CR = []
for t in WINDOWS:
    r = ref[ref.window == t]
    r = r[[(t, c) in common for c in CELL[r.tilenum.to_numpy() - 1]]]
    W = fov_R(r.lat, r.lon)
    for lab in RUNS:
        omf = r.obs.to_numpy() - W @ FCST[lab, t]
        CR.append({'window': t, 'run': lab, 'n': len(r), 'mean': omf.mean(), 'std': omf.std(),
                   'rms': np.sqrt(np.mean(omf ** 2))})
CR = pd.DataFrame(CR)
fig, axs = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
plot_runs = ['raw (OL)', 'super-obs, OL sc.', 'thin 25 km']
for lab in plot_runs:
    c = CR[CR.run == lab]
    axs[0].plot(np.arange(len(c)), c['std'], 'o-', color=RCOL[lab], label=lab)
    axs[1].plot(np.arange(len(c)), c['mean'], 'o-', color=RCOL[lab], label=lab)
for ax, yl in zip(axs, ['std(O − F) [m³/m³]', 'mean(O − F) [m³/m³]']):
    ax.set_xticks(np.arange(len(WINDOWS)), [t.strftime('%m-%d\n%Hz') for t in WINDOWS], fontsize=7)
    ax.set_ylabel(yl); ax.grid(alpha=0.3)
axs[0].legend()
fig.suptitle('raw H SAF obs (OL scaling) against the OL, OL-scaled super-obs, and thinning forecasts', fontsize=12)
fig.savefig(FIG_DIR / 'common_reference_omf.png', dpi=150)
plt.show()
CR['day'] = CR.window.dt.strftime('%m-%d')
print(CR.groupby(['day', 'run'], sort=False)[['n', 'mean', 'std', 'rms']].mean()
        .to_string(float_format=lambda v: f'{v:.4f}' if abs(v) < 10 else f'{v:,.0f}'))

**Takeaway:** on 5/1 03z all runs are identical (shared forecast), a check. By 5/10, the OL-scaled super-ob run and
the thinning run fit the raw obs equally well (std 0.0263 and 0.0264, mean 0.0005 and 0.0006), both much better than
the OL (0.0299, mean 0.0029); the old-scaling run hardly improved on the OL (0.0292) and drifted wet-biased in the
OL's scaling (mean 0.0043). With the scaling consistent, this comparison no longer favours thinning by construction
(both runs assimilate these obs in this scaling, averaged or subsampled), and the two are indistinguishable.

### 7.2 Independent obs: SMAP and SMOS brightness temperatures

SMAP L1C and SMOS Tb are monitored (not assimilated) in all runs, so the same obs, in the same scaling, are
predicted from each run's forecast through the radiative transfer model. They are independent of H SAF, and the
fairest obs-space comparison available here. Obs are matched across runs by (window, species, tile) and kept only if
the obs value agrees.

In [ ]:
TB = {}
for lab, run in RUNS.items():
    parts = []
    for t in WINDOWS:
        f = sorted(RUN_DIR[run].glob(f'*ObsFcstAna.{t:%Y%m%d_%H}00z.nc4'))[0]
        with nc.Dataset(f) as d:
            v = d['obsparam_descr']
            descr = [str(x).strip() for x in (nc.chartostring(v[:]) if v.ndim > 1 else v[:])]
            df = pd.DataFrame({k: np.asarray(d[k][:]) for k in ['species', 'tilenum', 'obs', 'fcst', 'fcstvar', 'obsvar']})
        tb_sp = {k + 1: s for k, s in enumerate(descr) if s.startswith(('SMAP_L1C_Tb', 'SMOS_fit_Tb'))}
        df = df[df.species.isin(tb_sp)].copy()
        df['sensor'] = df.species.map(tb_sp).str.split('_').str[0]
        df['window'] = t
        parts.append(df)
    TB[lab] = pd.concat(parts, ignore_index=True)
key = ['window', 'species', 'tilenum']
tbm = TB['raw (OL)'][key + ['sensor', 'obs', 'fcst']].rename(columns={'fcst': 'F_ol'})
TB_COL = [('raw (OL)', 'F_ol'), ('super-obs', 'F_sob'), ('super-obs, OL sc.', 'F_sobol'), ('thin 25 km', 'F_thin')]
for lab, suf in [('super-obs', 'sob'), ('super-obs, OL sc.', 'sobol'), ('thin 25 km', 'thin')]:
    tbm = tbm.merge(TB[lab][key + ['obs', 'fcst']].rename(columns={'obs': f'obs_{suf}', 'fcst': f'F_{suf}'}), on=key)
same = ((np.abs(tbm.obs - tbm.obs_sob) < 1e-3) & (np.abs(tbm.obs - tbm.obs_thin) < 1e-3)
        & (np.abs(tbm.obs - tbm.obs_sobol) < 1e-3))
print(f'{len(tbm):,} Tb obs in all four runs, {same.mean():.2%} with identical obs value')
tbm = tbm[same]
tbm['day'] = tbm.window.dt.strftime('%m-%d')
# where H SAF DA changed the forecast Tb: either DA run's forecast differs from the OL's by more than 1 K
# (selected on the forecasts, not on the obs)
tbm['subset'] = np.where(np.abs(tbm[['F_sob', 'F_sobol', 'F_thin']].to_numpy() - tbm.F_ol.to_numpy()[:, None]).max(1) > 1.0,
                         'DA changed F by > 1 K', 'all')
tbm = pd.concat([tbm.assign(subset='all'), tbm[tbm.subset != 'all']], ignore_index=True)
rows = []
for (day, sensor, subset), g in tbm.groupby(['day', 'sensor', 'subset'], sort=False):
    for lab, c in TB_COL:
        omf = g.obs - g[c]
        rows.append({'day': day, 'sensor': sensor, 'subset': subset, 'run': lab, 'n': len(g), 'mean(O − F) [K]': omf.mean(),
                     'std(O − F) [K]': omf.std(), 'rms [K]': np.sqrt(np.mean(omf ** 2))})
TBS = pd.DataFrame(rows).set_index(['day', 'sensor', 'subset', 'run']).sort_index(level=[0, 1, 2], sort_remaining=False)
print(TBS.to_string(float_format=lambda v: f'{v:.3f}' if abs(v) < 1000 else f'{v:,.0f}'))

# per window, SMAP only
fig, axs = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True)
g = tbm[(tbm.sensor == 'SMAP') & (tbm.subset == 'all')]
for lab, c in TB_COL:
    if lab == 'super-obs':
        continue
    s_ = (g.obs - g[c]).groupby(g.window)
    w = s_.std().index
    x = [WINDOWS.index(t) for t in w]
    axs[0].plot(x, s_.std().to_numpy(), 'o-', color=RCOL[lab], label=lab)
    axs[1].plot(x, s_.mean().to_numpy(), 'o-', color=RCOL[lab], label=lab)
for ax, yl in zip(axs, ['std(O − F) [K]', 'mean(O − F) [K]']):
    ax.set_xticks(np.arange(len(WINDOWS)), [t.strftime('%m-%d\n%Hz') for t in WINDOWS], fontsize=7)
    ax.set_ylabel(yl); ax.grid(alpha=0.3)
axs[0].legend()
fig.suptitle('SMAP L1C Tb (monitored in all runs) against the OL, OL-scaled super-obs, and thinning forecasts', fontsize=12)
fig.savefig(FIG_DIR / 'smap_tb_omf.png', dpi=150)
plt.show()

**Takeaway:** against independent Tb the differences are small: H SAF DA changes the forecast Tb by more than 1 K on
only ~20–40% of the Tb obs, and there by hundredths to tenths of a K in O − F std. On those obs on 5/10, the OL-scaled
super-ob run and the thinning run are level with each other and with the OL in std (SMAP 8.469 and 8.467 vs 8.473 K)
and slightly better in rms (8.491 and 8.492 vs 8.527 K), mostly from a smaller mean O − F; for SMOS both are a few
hundredths of a K worse than the OL. Only the old-scaling super-ob run is clearly worse (SMAP std 8.604 K, SMOS 9.669
K). After ten days, with consistent scaling, neither method harms or helps the fit to Tb noticeably, and they do not
differ.

## 8. Summary

- **The runs did what the offline operators say.** Every assimilated super-ob and thinned obs is reproduced offline
  at the same position, and the footprints reproduce `fcst` to ~1e-4 m³/m³, so the maps of footprints and tile weights
  are the ones GEOSldas used.
- **The obs scaling, not the reduction method, made the first super-ob run different.** Scaling derived for the old
  super-obs added ~0.015 m³/m³ of tile-to-tile noise to the new super-obs; that run made larger, coherent increments,
  overshot its obs over SW Kansas (A − F 2.3× O − F), drifted wet-biased against the raw obs and fitted SMAP/SMOS Tb
  worse. Rerun with the raw-obs (OL) scaling, all of that disappears.
- **With consistent scaling, super-obs and thinning are indistinguishable** after ten days: the same fit to the raw
  H SAF obs (O − F std 0.0263 vs 0.0264 on 5/10, both well below the OL's 0.0299), the same fit to independent SMAP
  and SMOS Tb (within 0.01–0.02 K), and no overshoot in either. Super-obs make somewhat smaller increments (rms 0.011
  vs 0.013 on 5/10) for the same fit, from their slightly less noisy O − F.
- **Obs and footprints.** Per window: raw ~80–86k obs, super-obs ~29–32k (3 raw obs each, 20 km apart, half with a
  neighbour < 20 km), thinning ~21–23k (single raw obs, ≥ 25 km apart). The median tile sits in 8 raw-mode footprints,
  4 super-ob footprints and 2 thinned ones; super-obs cover the area most evenly.
- **Where the obs fall.** Thinning keeps a different set of grid points every pass (only 37% of thinned obs repeat the
  previous pass's location), so the weight a tile gets varies by ~17% between passes and ~1 in 20 tile-passes inside
  the swath is nearly unobserved; super-obs sit at the same position in a fully covered cell every pass, so every tile
  gets the same weight. Thinning acts as a fresh random subsample each cycle; super-obs are a fixed operator.
- **What this leaves for the choice:** on skill, nothing yet separates them; super-obs give smoother increments and
  use all raw obs, at the cost of an operator that must match its scaling (a super-ob climatology, or the raw-obs
  scaling as here) and the grid-collision and tile-spacing logic; thinning is simpler and works with the raw-obs
  scaling as is. A longer run and in-situ evaluation are needed before skill can decide.
- **Limits:** two days of 10-day H SAF-only runs, ensemble-mean output; in-situ skill and longer runs are next.

In [ ]:
print(f'runtime {time.time() - T_START:.0f} s')